In [0]:

# ============================================================
# CELDA 1. PREPARAR DATOS PARA XGBOOST CON CLIMA
# ============================================================

# Objetivo:
# Predecir si un vuelo tendrá un retraso de salida
# de 15 minutos o más, utilizando información
# programada del vuelo y pronósticos meteorológicos.
#
# Los datos de clima provienen de Open-Meteo previous_day1.
# Usamos la misma muestra del modelo sin clima.

from pyspark.sql import functions as F

TABLA_GOLD = "workspace.gold_flights.flights_weather_features"

vuelos = spark.table(TABLA_GOLD)

# ------------------------------------------------------------
# 1. VARIABLES DEL MODELO
# ------------------------------------------------------------

# Características operativas disponibles antes del vuelo.
variables_operativas = [
    "origin",
    "dest",
    "airline",
    "month",
    "day_of_week",
    "departure_hour",
    "is_weekend",
    "is_summer",
    "is_peak_hour"
]

# Pronósticos meteorológicos del día anterior.
variables_clima = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

objetivo = "delayed_15"

# Conservamos fecha para dividir los periodos.
datos = vuelos.select(
    "flight_date",
    *variables_operativas,
    *variables_clima,
    objetivo
)

# ------------------------------------------------------------
# 2. MISMA MUESTRA REPRODUCIBLE DEL BASELINE
# ------------------------------------------------------------

# Esta regla es idéntica a la utilizada por José Miguel.
# Por eso, ambos modelos evaluarán los mismos vuelos.

datos = datos.filter(
    F.pmod(
        F.xxhash64(
            "flight_date",
            "origin",
            "dest",
            "airline",
            "departure_hour"
        ),
        F.lit(1000)
    ) < 30
)

# ------------------------------------------------------------
# 3. MISMA DIVISIÓN CRONOLÓGICA
# ------------------------------------------------------------

train = datos.filter(
    F.col("flight_date") < F.lit("2025-06-01")
)

validation = datos.filter(
    (F.col("flight_date") >= F.lit("2025-06-01")) &
    (F.col("flight_date") < F.lit("2025-09-01"))
)

test = datos.filter(
    F.col("flight_date") >= F.lit("2025-09-01")
)

# ------------------------------------------------------------
# 4. VALIDAR TAMAÑOS Y VARIABLES METEOROLÓGICAS
# ------------------------------------------------------------

print("=" * 55)
print("PREPARACIÓN DEL XGBOOST CON CLIMA")
print("=" * 55)

for nombre, conjunto in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Prueba", test)
]:
    resumen = conjunto.agg(
        F.count("*").alias("total"),
        F.round(
            F.avg("delayed_15") * 100,
            2
        ).alias("delay_rate")
    ).first()

    print(
        f"{nombre}: "
        f"{resumen['total']:,} vuelos | "
        f"{resumen['delay_rate']}% retrasados"
    )

# Comprobamos que las variables meteorológicas estén completas.
faltantes = datos.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in variables_clima
]).first().asDict()

print("\nValores faltantes en clima:")
print(faltantes)

if any(cantidad > 0 for cantidad in faltantes.values()):
    raise ValueError(
        "Hay datos meteorológicos faltantes. Revisar antes de entrenar."
    )

print("\nDATOS CON CLIMA PREPARADOS CORRECTAMENTE.")


In [0]:

%pip install xgboost scikit-learn scipy


In [0]:

dbutils.library.restartPython()


In [0]:

# ============================================================
# CELDA 2. ENTRENAR XGBOOST CON PRONÓSTICOS METEOROLÓGICOS
# ============================================================

# Comparamos el modelo con clima contra el baseline.
# Utilizamos exactamente los mismos vuelos, fechas
# e hiperparámetros. Solo agregamos variables del clima.
#
# No incluimos dep_delay como variable predictora.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. CONVERTIR LOS CONJUNTOS A PANDAS
# ------------------------------------------------------------

# Solo convertimos la muestra de aproximadamente
# 259 mil vuelos, no los millones de registros Gold.

train_pd = train.toPandas()
val_pd = validation.toPandas()
test_pd = test.toPandas()

print("Entrenamiento:", len(train_pd))
print("Validación:", len(val_pd))
print("Prueba:", len(test_pd))

# ------------------------------------------------------------
# 2. DEFINIR CARACTERÍSTICAS
# ------------------------------------------------------------

categoricas = [
    "origin",
    "dest",
    "airline"
]

numericas = [
    "month",
    "day_of_week",
    "departure_hour",
    "is_weekend",
    "is_summer",
    "is_peak_hour",
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "relative_humidity_2m",
    "cloud_cover",
    "pressure_msl"
]

X_train = train_pd[categoricas + numericas]
X_val = val_pd[categoricas + numericas]
X_test = test_pd[categoricas + numericas]

y_train = train_pd["delayed_15"].astype(int)
y_val = val_pd["delayed_15"].astype(int)
y_test = test_pd["delayed_15"].astype(int)

# ------------------------------------------------------------
# 3. CODIFICAR AEROPUERTOS Y AEROLÍNEAS
# ------------------------------------------------------------

# El codificador aprende únicamente de entrenamiento.

try:
    encoder_clima = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )
except TypeError:
    encoder_clima = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True
    )

encoder_clima.fit(X_train[categoricas])

def preparar_variables_clima(df):
    categorias = encoder_clima.transform(
        df[categoricas]
    )

    numeros = csr_matrix(
        df[numericas]
        .astype("float32")
        .to_numpy()
    )

    return hstack(
        [categorias, numeros],
        format="csr",
        dtype=np.float32
    )

X_train_final = preparar_variables_clima(X_train)
X_val_final = preparar_variables_clima(X_val)
X_test_final = preparar_variables_clima(X_test)

print("Variables codificadas:", X_train_final.shape[1])

# ------------------------------------------------------------
# 4. ENTRENAR XGBOOST
# ------------------------------------------------------------

# Mismos hiperparámetros que el modelo sin clima.

modelo_clima = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

modelo_clima.fit(
    X_train_final,
    y_train,
    eval_set=[(X_val_final, y_val)],
    verbose=False
)

print("\nXGBOOST CON CLIMA ENTRENADO CORRECTAMENTE")

# ------------------------------------------------------------
# 5. SELECCIONAR UMBRAL CON VALIDACIÓN
# ------------------------------------------------------------

# No copiamos automáticamente el umbral del baseline.
# Buscamos el mejor F1 utilizando solo validación.

proba_val = modelo_clima.predict_proba(X_val_final)[:, 1]
proba_test = modelo_clima.predict_proba(X_test_final)[:, 1]

resultados_umbrales = []

for umbral in np.arange(0.10, 0.51, 0.025):
    pred = (proba_val >= umbral).astype(int)

    resultados_umbrales.append({
        "umbral": round(float(umbral), 3),
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        )
    })

tabla_umbrales = pd.DataFrame(resultados_umbrales)

mejor_fila = tabla_umbrales.loc[
    tabla_umbrales["f1"].idxmax()
]

mejor_umbral_clima = float(mejor_fila["umbral"])

print("\nMEJORES UMBRALES EN VALIDACIÓN")
display(
    tabla_umbrales
    .sort_values("f1", ascending=False)
    .head(10)
    .round(4)
)

print("Mejor umbral:", mejor_umbral_clima)

# ------------------------------------------------------------
# 6. EVALUAR EL CONJUNTO DE PRUEBA
# ------------------------------------------------------------

# Aplicamos el umbral elegido en validación.
# No modificamos el umbral con resultados de prueba.

pred_test_clima = (
    proba_test >= mejor_umbral_clima
).astype(int)

metricas_clima = pd.DataFrame([{
    "Modelo": "XGBoost con clima",
    "Umbral": mejor_umbral_clima,
    "Accuracy": accuracy_score(
        y_test, pred_test_clima
    ),
    "Precision": precision_score(
        y_test, pred_test_clima, zero_division=0
    ),
    "Recall": recall_score(
        y_test, pred_test_clima, zero_division=0
    ),
    "F1": f1_score(
        y_test, pred_test_clima, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test, proba_test
    )
}])

print("\nMÉTRICAS FINALES DEL XGBOOST CON CLIMA")
display(metricas_clima.round(4))

# ------------------------------------------------------------
# 7. MATRIZ DE CONFUSIÓN
# ------------------------------------------------------------

matriz_clima = confusion_matrix(
    y_test, pred_test_clima
)

ConfusionMatrixDisplay(
    confusion_matrix=matriz_clima,
    display_labels=["Sin retraso", "Retraso >=15 min"]
).plot(values_format="d", cmap="Blues")

plt.title("XGBoost con clima - Matriz de confusión")
plt.tight_layout()
plt.show()

print("\nXGBOOST CON CLIMA COMPLETADO.")


In [0]:

# ============================================================
# CELDA 3. GUARDAR RESULTADOS DEL XGBOOST CON CLIMA
# ============================================================

# Guardamos las métricas finales del conjunto de prueba.
# El umbral fue seleccionado utilizando validación.

metricas_guardar = metricas_clima.copy()

metricas_guardar.columns = [
    "model",
    "threshold",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

# Mostrar las métricas de forma clara
print("=" * 55)
print("RESULTADOS FINALES - XGBOOST CON CLIMA")
print("=" * 55)

for columna in [
    "threshold", "accuracy", "precision",
    "recall", "f1", "roc_auc"
]:
    valor = float(metricas_guardar.iloc[0][columna])
    print(f"{columna}: {valor:.4f}")

# Guardar resultados en Delta
df_metricas = spark.createDataFrame(metricas_guardar)

(
    df_metricas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_flights.weather_model_metrics"
    )
)

print("\nMÉTRICAS DEL MODELO CON CLIMA GUARDADAS CORRECTAMENTE")


In [0]:

# ============================================================
# CELDA 4. COMPARACIÓN DE LOS DOS MODELOS XGBOOST
# ============================================================

# Comparamos los resultados del conjunto de prueba.
# Ambos modelos utilizaron los mismos vuelos y fechas.
# Los umbrales se seleccionaron con validación.

from pyspark.sql import functions as F

baseline = spark.table(
    "workspace.gold_flights.baseline_model_metrics"
)

con_clima = spark.table(
    "workspace.gold_flights.weather_model_metrics"
)

comparacion = baseline.unionByName(con_clima)

print("=" * 55)
print("COMPARACIÓN FINAL DE MODELOS")
print("=" * 55)

display(
    comparacion.select(
        "model",
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    )
)

# Diferencias: modelo con clima menos modelo sin clima
a = baseline.first()
b = con_clima.first()

print("\nDIFERENCIAS DEL MODELO CON CLIMA RESPECTO AL BASELINE")

for metrica in [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]:
    diferencia = float(b[metrica]) - float(a[metrica])
    print(f"{metrica}: {diferencia:+.4f}")

print("\nCOMPARACIÓN COMPLETADA.")
